# Notebook 03 — Parser Validation

The purpose of this notebook is to test and improve the Gol.gg
game-page parser across multiple games.

We will:

1. Load saved game HTML files.
2. Extract game-level information.
3. Identify blue and red teams.
4. Identify the winning team and winning side.
5. Extract timeline events.
6. Validate extraction results across multiple games.
7. Record parsing errors and unexpected HTML structures.

The parser should be reusable across different games and tournaments.

In [1]:
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

In [2]:
# Find the project root by searching the current directory
# and its parent directories for data/raw/html.

current_dir = Path.cwd()

project_root = None

for folder in [current_dir, *current_dir.parents]:

    if (folder / "data" / "raw" / "html").exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError("Could not locate data/raw/html")

# Define our input and output folders
html_dir = project_root / "data" / "raw" / "html"
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("HTML directory:", html_dir)
print("Processed directory:", processed_dir)

Project root: c:\Users\boyle\League_Of_Legends_Herald_Analysis
HTML directory: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw\html
Processed directory: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\processed


In [3]:
#check how many game pages we have

html_files = sorted(html_dir.glob("game_*.html"))

print("Number of saved game pages:", len(html_files))

for file in html_files:
    print(file.name)

Number of saved game pages: 1
game_83713.html


In [4]:
#inspect team headers for winning team

# Load our first saved game
test_file = html_files[0]

with open(test_file, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "lxml")

# Inspect blue and red team headers
for class_name in ["blue-line-header", "red-line-header"]:

    header = soup.find("div", class_=class_name)

    print(f"\n{class_name}:")

    if header:
        print(header.get_text(" ", strip=True))
    else:
        print("Header not found")


blue-line-header:
Gen.G - WIN

red-line-header:
Invictus Gaming - LOSS


In [5]:
#extract winner

def extract_game_teams(soup):

    # Find the blue and red team headers
    blue_header = soup.find("div", class_="blue-line-header")
    red_header = soup.find("div", class_="red-line-header")

    # Check that both headers exist
    if blue_header is None or red_header is None:
        raise ValueError("Could not find both team headers")

    # Extract team names and header text
    blue_text = blue_header.get_text(" ", strip=True)
    red_text = red_header.get_text(" ", strip=True)

    # Determine whether each side won
    blue_won = bool(re.search(r"\bWIN\b", blue_text, re.IGNORECASE))
    red_won = bool(re.search(r"\bWIN\b", red_text, re.IGNORECASE))

    # Extract team names from the links
    blue_link = blue_header.find("a")
    red_link = red_header.find("a")

    if blue_link is None or red_link is None:
        raise ValueError("Could not find team name links")

    blue_team = blue_link.get_text(strip=True)
    red_team = red_link.get_text(strip=True)

    # Determine the winning team and side
    if blue_won and not red_won:
        winning_team = blue_team
        winning_side = "blue"

    elif red_won and not blue_won:
        winning_team = red_team
        winning_side = "red"

    else:
        raise ValueError(
            f"Could not uniquely identify winner: "
            f"blue={blue_text}, red={red_text}"
        )

    return {
        "blue_team": blue_team,
        "red_team": red_team,
        "winning_team": winning_team,
        "winning_side": winning_side
    }

In [6]:
#test winner extraction

game_info = extract_game_teams(soup)

game_info

{'blue_team': 'Gen.G',
 'red_team': 'Invictus Gaming',
 'winning_team': 'Gen.G',
 'winning_side': 'blue'}

## 8. Build a reusable game-page parser

Create a function that extracts both game-level information
and timeline events from a saved Gol.gg game page.

The function should work with different game IDs and teams.

In [11]:

def parse_game_page(html, game_id):
    """
    Parse a Gol.gg game page and extract:
    1. Blue and red team names
    2. Winning team and winning side
    3. Timeline team abbreviations
    4. Timeline events, timestamps, and event ownership

    Returns:
        game_info: Dictionary containing game metadata
        events: List of dictionaries containing timeline events
    """

    # --------------------------------------------------
    # 1. Parse the HTML
    # --------------------------------------------------

    soup = BeautifulSoup(html, "lxml")

    # --------------------------------------------------
    # 2. Extract team names from the game headers
    # --------------------------------------------------

    blue_header = soup.find("div", class_="blue-line-header")
    red_header = soup.find("div", class_="red-line-header")

    if blue_header is None or red_header is None:
        raise ValueError("Could not find both blue and red team headers")

    blue_link = blue_header.find("a")
    red_link = red_header.find("a")

    if blue_link is None or red_link is None:
        raise ValueError("Could not find team name links")

    blue_team = blue_link.get_text(strip=True)
    red_team = red_link.get_text(strip=True)

    # --------------------------------------------------
    # 3. Identify the winning team
    # --------------------------------------------------

    blue_text = blue_header.get_text(" ", strip=True)
    red_text = red_header.get_text(" ", strip=True)

    blue_won = bool(re.search(r"\bWIN\b", blue_text, re.IGNORECASE))
    red_won = bool(re.search(r"\bWIN\b", red_text, re.IGNORECASE))

    if blue_won and not red_won:
        winning_team = blue_team
        winning_side = "blue"

    elif red_won and not blue_won:
        winning_team = red_team
        winning_side = "red"

    else:
        raise ValueError(
            f"Could not uniquely identify winner: "
            f"blue={blue_text}, red={red_text}"
        )

    # Store the game-level information
    game_info = {
        "game_id": str(game_id),
        "blue_team": blue_team,
        "red_team": red_team,
        "winning_team": winning_team,
        "winning_side": winning_side
    }

    # --------------------------------------------------
    # 4. Find the timeline container
    # --------------------------------------------------

    timeline_container = None

    for div in soup.find_all("div"):

        classes = div.get("class", [])

        if (
            "col-12" in classes
            and "flex-wrap" in classes
            and div.find(class_="title_blue")
            and div.find(class_="title_red")
        ):
            timeline_container = div
            break

    if timeline_container is None:
        raise ValueError("Could not find the Gold graph & Timeline container")

    # --------------------------------------------------
    # 5. Extract timeline team labels
    # --------------------------------------------------

    timeline_blue = timeline_container.find(class_="title_blue")
    timeline_red = timeline_container.find(class_="title_red")

    if timeline_blue is None or timeline_red is None:
        raise ValueError("Could not find both timeline team labels")

    # These are abbreviations, such as GEN and IG.
    # They are not necessarily the same as the full team names.

    timeline_blue_label = timeline_blue.get_text(strip=True)
    timeline_red_label = timeline_red.get_text(strip=True)

    # Add timeline abbreviations to the game metadata.
    game_info["blue_team_timeline_label"] = timeline_blue_label
    game_info["red_team_timeline_label"] = timeline_red_label

    # --------------------------------------------------
    # 6. Map timeline event classes to full team names
    # --------------------------------------------------

    team_names = {
        "blue_action": blue_team,
        "red_action": red_team
    }

    # --------------------------------------------------
    # 7. Extract timeline events
    # --------------------------------------------------

    events = []

    # Get the direct child spans.
    # The first span contains the team labels.
    timeline_spans = timeline_container.find_all(
        "span",
        recursive=False
    )

    # Skip the team-label span.
    event_wrappers = timeline_spans[1:]

    for wrapper in event_wrappers:

        # Each wrapper can contain a blue lane and a red lane.
        lanes = wrapper.find_all(
            "span",
            recursive=False
        )

        for lane in lanes:

            classes = lane.get("class", [])

            # Determine whether this event belongs to blue or red.
            side_class = next(
                (side for side in team_names if side in classes),
                None
            )

            if side_class is None:
                continue

            # Find the event icon.
            img = lane.find("img")

            if img is None:
                continue

            # Extract the timestamp from the lane's text.
            timestamp_match = re.search(
                r"(?<!\d)(\d{1,2}:\d{2})(?!\d)",
                lane.get_text(" ", strip=True)
            )

            if timestamp_match is None:
                continue

            timestamp = timestamp_match.group(1)

            # Convert timestamp into seconds.
            minutes, seconds = map(int, timestamp.split(":"))

            event_time_seconds = minutes * 60 + seconds

            # Determine the side.
            side = (
                "blue"
                if side_class == "blue_action"
                else "red"
            )

            # Save the event.
            events.append({
                "game_id": str(game_id),
                "event_type": img.get("alt"),
                "team": team_names[side_class],
                "side": side,
                "event_time": timestamp,
                "event_time_seconds": event_time_seconds,
                "icon_src": img.get("src")
            })

    # --------------------------------------------------
    # 8. Sort events chronologically
    # --------------------------------------------------

    events.sort(
        key=lambda event: event["event_time_seconds"]
    )

    # Assign event numbers in chronological order.
    for number, event in enumerate(events, start=1):
        event["event_number"] = number

    # --------------------------------------------------
    # 9. Return results
    # --------------------------------------------------

    return game_info, events


In [12]:
#Test resuable parser on game 83713

# Load the first saved game
test_file = html_files[0]

with open(test_file, "r", encoding="utf-8") as f:
    test_html = f.read()

# Extract the game ID from the filename
game_id = re.search(r"game_(\d+)", test_file.stem).group(1)

# Run the parser
game_info, game_events = parse_game_page(test_html, game_id)

print("GAME INFORMATION")
print(game_info)

print("\nNUMBER OF EVENTS")
print(len(game_events))

GAME INFORMATION
{'game_id': '83713', 'blue_team': 'Gen.G', 'red_team': 'Invictus Gaming', 'winning_team': 'Gen.G', 'winning_side': 'blue', 'blue_team_timeline_label': 'GEN', 'red_team_timeline_label': 'IG'}

NUMBER OF EVENTS
9
